# Chapter 6: Hypothesis testing

**Data Analysis with Python for Health Specialists** | 3 hours

---

## 1. The logic of hypothesis testing

- **Null hypothesis (H0):** No effect, no difference, no association.
- **Alternative hypothesis (H1):** There *is* an effect, difference, or association.

We collect data, compute a test statistic, and ask: *if H0 were true, how likely is it that we would see data this extreme?* That probability is the **p-value**.

### The steps of a hypothesis test
1. State the hypotheses (H0 and H1).
2. Choose the significance level alpha (usually 0.05).
3. Collect data and compute the test statistic.
4. Compute the p-value.
5. Decide: if p < alpha, reject H0.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

## 2. p-values: what they mean and what they don't

A p-value is the probability of observing data as extreme as yours, *assuming H0 is true*.

What a p-value is **NOT**:
- NOT the probability that H0 is true.
- NOT the probability that the result is due to chance.
- Does NOT measure the size of the effect.
- p = 0.049 is not meaningfully different from p = 0.051.

> **Warning:** A small p-value does not mean a *large* or *clinically important* effect. A study with 100,000 patients can find p < 0.001 for a BP difference of 0.5 mmHg. Always report **effect size** alongside p-values.

In [ ]:
# Visualize the logic: sampling distribution under H0
np.random.seed(42)
null_distribution = np.random.normal(loc=0, scale=1, size=10000)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(null_distribution, bins=60, density=True, alpha=0.7, color="steelblue")
ax.axvline(1.96, color="red", linestyle="--", label="Critical value (alpha=0.05)")
ax.axvline(-1.96, color="red", linestyle="--")
ax.fill_betweenx([0, 0.45], 1.96, 3.5, alpha=0.3, color="red", label="Rejection region")
ax.fill_betweenx([0, 0.45], -3.5, -1.96, alpha=0.3, color="red")
ax.set_xlabel("Test statistic (z)")
ax.set_ylabel("Density")
ax.set_title("Two-tailed test: rejection regions under H0")
ax.legend()
plt.tight_layout()
plt.show()

## 3. One-sample t-test

**Question:** Is the mean total cholesterol in our clinic different from the national average of 200 mg/dL?

In [ ]:
# Simulated clinic cholesterol data (mg/dL)
np.random.seed(42)
cholesterol = np.random.normal(loc=212, scale=35, size=80)

# One-sample t-test: is the mean different from 200?
t_stat, p_value = stats.ttest_1samp(cholesterol, popmean=200)
print(f"Sample mean: {cholesterol.mean():.1f} mg/dL")
print(f"t-statistic: {t_stat:.3f}")
print(f"p-value: {p_value:.4f}")

if p_value < 0.05:
    print("Reject H0: mean cholesterol differs from 200 mg/dL")
else:
    print("Fail to reject H0: no evidence mean differs from 200 mg/dL")

## 4. Two-sample t-test

**Question:** Is systolic blood pressure different between the treatment group and control group?

In [ ]:
# Simulated clinical trial data
np.random.seed(42)
treatment = np.random.normal(loc=128, scale=15, size=60)
control = np.random.normal(loc=138, scale=16, size=55)

# Independent two-sample t-test
t_stat, p_value = stats.ttest_ind(treatment, control)
print(f"Treatment mean: {treatment.mean():.1f} mmHg")
print(f"Control mean:   {control.mean():.1f} mmHg")
print(f"Difference:     {control.mean() - treatment.mean():.1f} mmHg")
print(f"t-statistic:    {t_stat:.3f}")
print(f"p-value:        {p_value:.4f}")

### Checking assumptions

In [ ]:
# Normality: Shapiro-Wilk test (H0: data is normally distributed)
_, p_treat = stats.shapiro(treatment)
_, p_ctrl = stats.shapiro(control)
print(f"Shapiro-Wilk p (treatment): {p_treat:.4f}")
print(f"Shapiro-Wilk p (control):   {p_ctrl:.4f}")

# Equal variances: Levene's test
_, p_levene = stats.levene(treatment, control)
print(f"Levene's test p: {p_levene:.4f}")

# If variances are unequal, use Welch's t-test:
t_stat, p_value = stats.ttest_ind(treatment, control, equal_var=False)
print(f"Welch's t-test p-value: {p_value:.4f}")

## 5. Paired t-test

**Question:** Does a 12-week exercise intervention reduce systolic blood pressure?

Each patient is measured *before* and *after*. The paired t-test uses within-patient differences.

In [ ]:
# Simulated before/after intervention data
np.random.seed(42)
n_patients = 45
bp_before = np.random.normal(loc=142, scale=12, size=n_patients)
bp_after = bp_before - np.random.normal(loc=8, scale=6, size=n_patients)

# Paired t-test
t_stat, p_value = stats.ttest_rel(bp_before, bp_after)
differences = bp_before - bp_after
print(f"Mean BP before:     {bp_before.mean():.1f} mmHg")
print(f"Mean BP after:      {bp_after.mean():.1f} mmHg")
print(f"Mean reduction:     {differences.mean():.1f} mmHg")
print(f"t-statistic:        {t_stat:.3f}")
print(f"p-value:            {p_value:.6f}")

In [ ]:
# Visualize before/after
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Paired lines
for i in range(n_patients):
    axes[0].plot([0, 1], [bp_before[i], bp_after[i]],
                 color="gray", alpha=0.4)
axes[0].plot([0, 1], [bp_before.mean(), bp_after.mean()],
             color="red", linewidth=3, marker="o", markersize=8)
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(["Before", "After"])
axes[0].set_ylabel("Systolic BP (mmHg)")
axes[0].set_title("Individual patient trajectories")

# Distribution of differences
axes[1].hist(differences, bins=15, edgecolor="black", alpha=0.7, color="steelblue")
axes[1].axvline(0, color="red", linestyle="--", label="No change")
axes[1].axvline(differences.mean(), color="green", linestyle="--",
                label=f"Mean = {differences.mean():.1f}")
axes[1].set_xlabel("BP reduction (mmHg)")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Distribution of BP changes")
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Chi-square test of independence

**Question:** Is diabetes prevalence associated with obesity category?

In [ ]:
# Simulated cross-tabulation data
np.random.seed(42)
n = 500
bmi_cat = np.random.choice(["Normal", "Overweight", "Obese"],
                           size=n, p=[0.3, 0.4, 0.3])
# Diabetes probability increases with obesity
diabetes_prob = {"Normal": 0.08, "Overweight": 0.18, "Obese": 0.35}
diabetes = [np.random.binomial(1, diabetes_prob[b]) for b in bmi_cat]

df_chi = pd.DataFrame({"bmi_category": bmi_cat, "diabetes": diabetes})
df_chi["diabetes_label"] = df_chi["diabetes"].map({0: "No", 1: "Yes"})

# Contingency table
contingency = pd.crosstab(df_chi["bmi_category"], df_chi["diabetes_label"])
print(contingency)
print()

# Chi-square test
chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
print(f"Chi-square statistic: {chi2:.2f}")
print(f"Degrees of freedom:   {dof}")
print(f"p-value:              {p_value:.4f}")
print(f"\nExpected frequencies (if independent):")
print(pd.DataFrame(expected, index=contingency.index,
                   columns=contingency.columns).round(1))

## 7. Mann-Whitney U test

When data is not normally distributed (hospital length-of-stay, cost data, small samples), use the Mann-Whitney U test. It compares *ranks* rather than means.

In [ ]:
# Hospital length of stay: surgical vs non-surgical (right-skewed data)
np.random.seed(42)
los_surgical = np.random.exponential(scale=7, size=40)
los_medical = np.random.exponential(scale=4.5, size=45)

# Mann-Whitney U test
u_stat, p_value = stats.mannwhitneyu(los_surgical, los_medical,
                                      alternative="two-sided")
print(f"Median LOS (surgical): {np.median(los_surgical):.1f} days")
print(f"Median LOS (medical):  {np.median(los_medical):.1f} days")
print(f"U statistic:           {u_stat:.0f}")
print(f"p-value:               {p_value:.4f}")

In [ ]:
# Visualize the skewed distributions
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(los_surgical, bins=15, alpha=0.6, label="Surgical", color="coral")
ax.hist(los_medical, bins=15, alpha=0.6, label="Medical", color="steelblue")
ax.set_xlabel("Length of stay (days)")
ax.set_ylabel("Frequency")
ax.set_title("Hospital length of stay by department")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Multiple testing correction

If you test 20 hypotheses at alpha = 0.05, you expect 1 false positive even if nothing is going on.

In [ ]:
# Simulation: 1000 tests where H0 is true for all
np.random.seed(42)
n_tests = 1000
p_values = []
for _ in range(n_tests):
    a = np.random.normal(0, 1, 30)
    b = np.random.normal(0, 1, 30)
    _, p = stats.ttest_ind(a, b)
    p_values.append(p)

p_values = np.array(p_values)
false_positives = (p_values < 0.05).sum()
print(f"Tests performed: {n_tests}")
print(f"False positives (p < 0.05): {false_positives}")
print(f"False positive rate: {false_positives/n_tests:.1%}")

In [ ]:
from statsmodels.stats.multitest import multipletests

# Bonferroni correction
rejected_bonf, pvals_corrected_bonf, _, _ = multipletests(
    p_values, alpha=0.05, method="bonferroni"
)
print(f"Bonferroni: {rejected_bonf.sum()} significant results")

# Benjamini-Hochberg FDR correction
rejected_fdr, pvals_corrected_fdr, _, _ = multipletests(
    p_values, alpha=0.05, method="fdr_bh"
)
print(f"FDR (BH): {rejected_fdr.sum()} significant results")

## 9. Effect size: clinical vs statistical significance

### Cohen's d

- |d| < 0.2: negligible
- 0.2 <= |d| < 0.5: small
- 0.5 <= |d| < 0.8: medium
- |d| >= 0.8: large

In [ ]:
def cohens_d(group1, group2):
    """Compute Cohen's d for two independent groups."""
    n1, n2 = len(group1), len(group2)
    var1, var2 = group1.var(ddof=1), group2.var(ddof=1)
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    return (group1.mean() - group2.mean()) / pooled_std

# Using our treatment vs control BP data
d = cohens_d(treatment, control)
print(f"Cohen's d: {d:.2f}")
print(f"Interpretation: {'large' if abs(d) >= 0.8 else 'medium' if abs(d) >= 0.5 else 'small' if abs(d) >= 0.2 else 'negligible'} effect")

In [ ]:
# Large sample + tiny effect = significant p-value
np.random.seed(42)
huge_group1 = np.random.normal(120.0, 15, size=50000)
huge_group2 = np.random.normal(120.5, 15, size=50000)  # only 0.5 mmHg difference

t, p = stats.ttest_ind(huge_group1, huge_group2)
d = cohens_d(huge_group1, huge_group2)
print(f"Mean difference: {huge_group2.mean() - huge_group1.mean():.2f} mmHg")
print(f"p-value: {p:.6f}")
print(f"Cohen's d: {d:.3f}")
print("Statistically significant? Yes. Clinically meaningful? No.")

## 10. Practical: hypothesis testing with Framingham-style data

In [ ]:
# Load a Framingham-style cardiovascular dataset
url = ("https://raw.githubusercontent.com/dsrscientist/"
       "dataset-for-machine-learning/master/framingham.csv")
fram = pd.read_csv(url)
print(f"Shape: {fram.shape}")
print(f"Columns: {fram.columns.tolist()}")
fram.head()

In [ ]:
# Quick exploration
fram.describe()

In [ ]:
# Drop rows with missing values for simplicity
fram_clean = fram.dropna()
print(f"Clean dataset: {fram_clean.shape[0]} patients")

## Exercises

1. **One-sample t-test.** Is the mean total cholesterol different from the US national average of 200 mg/dL?
2. **Two-sample t-test.** Compare systolic BP between CHD (TenYearCHD=1) and non-CHD patients. Report p-value, mean difference, and Cohen's d.
3. **Chi-square test.** Create a 2x2 table of `currentSmoker` vs `TenYearCHD`. Is smoking associated with 10-year CHD risk?
4. **Mann-Whitney U.** Compare `cigsPerDay` between CHD and non-CHD groups. Why is Mann-Whitney more appropriate here?
5. **Multiple testing.** Run t-tests for all continuous variables (age, totChol, sysBP, diaBP, BMI, heartRate, glucose). Apply Bonferroni and FDR correction.
6. **Clinical interpretation.** For the variable with the largest effect size, discuss clinical meaningfulness.

In [ ]:
# Exercise 1: your code here


In [ ]:
# Exercise 2: your code here


In [ ]:
# Exercise 3: your code here


In [ ]:
# Exercise 4: your code here


In [ ]:
# Exercise 5: your code here


In [ ]:
# Exercise 6: your code here
